# 第17回　総合課題 — 手書き数字認識
***
> **前提**: 第16回で学習・保存した `best_mnist_model.pth` を使い，自作の手書き数字で推論します。
>
> **実行環境**: 手書き推論 UI は **Google Colab** 上での実行を推奨します（`google.colab.output` を使用）。Colab では第14〜16回を順に実行してから本課題に取り組んでください。ローカル Jupyter では問題1と問題3（考察）のみ実施可能です。

## 目次
1. モデルの読み込み
2. 手書き推論 UI（完成コード — 編集不要）
3. 推論結果の記録
4. 考察

In [ ]:
%pip install -q torch torchvision


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


## 問題1
***
第16回と同じ `SimpleCNN` クラスを定義し，`best_mnist_model.pth` から `load_state_dict` でモデルを復元してください．

テストデータでの正解率を再確認して出力してください．

#### Hints
```python
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleCNN().to(device)
model.load_state_dict(torch.load("best_mnist_model.pth", map_location=device))
model.eval()
```

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # ここにあなたのコードを書いてください

    def forward(self, x):
        # ここにあなたのコードを書いてください
        pass


# モデル読み込みとテスト正解率
# ここにあなたのコードを書いてください


## 手書き推論 UI
***
以下のセルは **完成コード** です。編集せず実行してください。

Canvas に 0〜9 の数字を書き，「推論する」ボタンを押すと予測結果が表示されます。

> Canvas は黒背景・白線です。MNIST は白背景・黒線のため，推論時に色を反転しています。

In [ ]:
# 手書き推論UI（完成コード — 編集不要）
import numpy as np
from IPython.display import display, HTML
import base64
from PIL import Image
import io
from google.colab import output


def predict_digit(img_data_url):
    header, data = img_data_url.split(",", 1)
    img_bytes = base64.b64decode(data)
    img = Image.open(io.BytesIO(img_bytes)).convert("L")

    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype("float32") / 255.0
    arr = 1.0 - arr  # Canvas=黒地白線 → MNIST=白地黒線 に反転

    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        digit = int(probs.argmax(dim=1).item())
        confidence = float(probs.max().item()) * 100
    return digit, confidence


def _on_predict(data_url):
    digit, conf = predict_digit(data_url)
    print(f"予測: {digit}  信頼度: {conf:.1f}%")


output.register_callback("predict_from_js", _on_predict)

html_code = """
<style>
  body { font-family: sans-serif; }
  #canvas {
    border: 3px solid #333;
    border-radius: 8px;
    cursor: crosshair;
    background: black;
    display: block;
    margin: 10px 0;
  }
  .btn {
    padding: 10px 24px;
    margin: 4px;
    font-size: 16px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
  }
  #predictBtn { background: #4CAF50; color: white; }
  #clearBtn   { background: #f44336; color: white; }
  #result {
    font-size: 32px;
    font-weight: bold;
    margin-top: 12px;
    min-height: 40px;
    color: #1a73e8;
  }
  #confidence { font-size: 16px; color: #555; }
</style>

<h3>数字を書いてください（0〜9）</h3>
<canvas id="canvas" width="280" height="280"></canvas>

<div>
  <button class="btn" id="predictBtn" onclick="predict()">推論する</button>
  <button class="btn" id="clearBtn"   onclick="clearCanvas()">クリア</button>
</div>

<div id="result">ここに結果が表示されます</div>
<div id="confidence"></div>

<script>
  const canvas = document.getElementById("canvas");
  const ctx    = canvas.getContext("2d");

  ctx.fillStyle = "black";
  ctx.fillRect(0, 0, 280, 280);
  ctx.strokeStyle = "white";
  ctx.lineWidth   = 20;
  ctx.lineCap     = "round";
  ctx.lineJoin    = "round";

  let drawing = false;
  let lastX = 0, lastY = 0;

  function getPos(e) {
    const rect = canvas.getBoundingClientRect();
    if (e.touches) {
      return {
        x: e.touches[0].clientX - rect.left,
        y: e.touches[0].clientY - rect.top
      };
    }
    return { x: e.clientX - rect.left, y: e.clientY - rect.top };
  }

  canvas.addEventListener("mousedown",  e => { drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("mousemove",  e => {
    if (!drawing) return;
    const p = getPos(e);
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(p.x, p.y);
    ctx.stroke();
    lastX = p.x; lastY = p.y;
  });
  canvas.addEventListener("mouseup",   () => drawing = false);
  canvas.addEventListener("mouseleave",() => drawing = false);

  canvas.addEventListener("touchstart",  e => { e.preventDefault(); drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchmove",   e => { e.preventDefault(); if (!drawing) return; const p = getPos(e); ctx.beginPath(); ctx.moveTo(lastX, lastY); ctx.lineTo(p.x, p.y); ctx.stroke(); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchend",    e => { e.preventDefault(); drawing = false; });

  function clearCanvas() {
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, 280, 280);
    document.getElementById("result").innerText = "ここに結果が表示されます";
    document.getElementById("confidence").innerText = "";
  }

  function predict() {
    const dataURL = canvas.toDataURL("image/png");
    google.colab.kernel.invokeFunction("predict_from_js", [dataURL], {});
  }
</script>
"""

display(HTML(html_code))
print("キャンバスを表示しました。数字を書いて「推論する」を押してください。")


## 問題2
***
上の UI で **0〜9 を各1回以上** 書き，予測結果を表形式で記録してください（正解ラベル / 予測 / 信頼度）．

誤認識があった場合，どの数字をどう間違えたかメモしてください．

#### Hints
```python
import pandas as pd
results = [
    {"正解": 3, "予測": 3, "信頼度": 98.5},
    {"正解": 7, "予測": 1, "信頼度": 62.3},
    # ... 0〜9 まで
]
df_results = pd.DataFrame(results)
print(df_results)
```

In [ ]:
# 推論結果の記録
# ここにあなたのコードを書いてください


## 問題3
***
第16回の結果と今回の推論を踏まえ，以下について述べてください（print または markdown セル追加可）．

1. 第16回で CNN が MLP より良かった理由
2. 自作数字で誤認識した場合の原因（線の太さ，位置，前処理の色反転など）
3. 精度をさらに上げるために試せること（1〜2個）


In [ ]:
# 考察
# ここにあなたのコードを書いてください
